In [11]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [12]:
def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [13]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,0.415121,39.293238,13.819007,18.370000,0.747024,6.460530,-6.460530,8.648361,8.648361,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,0.720893,36.652103,17.124485,29.821070,1.341544,13.932911,-13.932911,10.385726,10.385726,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,0.813072,38.321956,22.500644,35.111530,1.485426,20.615650,-20.615650,13.878609,13.878609,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,0.707903,33.359403,25.236451,26.727384,1.382853,20.219316,-20.219316,14.621450,14.621450,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,0.671389,32.548385,21.987350,24.789544,1.295489,16.746035,-16.746035,12.926420,12.926420,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.975153,14.998988,12.413334,18.013558,1.492962,14.908227,-14.908227,9.985670,9.985670,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,1.068629,13.720627,21.347952,18.137762,1.714749,28.220582,-28.220582,16.457558,16.457558,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.779983,11.527504,17.911727,11.147219,1.459002,17.320831,-17.320831,11.871701,11.871701,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,1.205378,12.385137,7.635535,18.450361,2.161599,11.374792,-11.374792,5.262211,5.262211,0.008310


In [ ]:
class HybridEnsemble:
    def __init__(self, use_stacking=False):
        self.use_stacking = use_stacking
        self.models = {}
        self.weights = {}
        self.model_scores = {}
        
    def _create_models(self):
        models = {
            'lgbm_1': LGBMRegressor(
                boosting_type='gbdt',
                colsample_bytree=0.95,
                learning_rate=0.08,
                max_bin=100,
                max_depth=12,
                metric='mape',
                min_child_samples=60,
                min_data_in_leaf=20,
                n_estimators=7000,
                num_leaves=50,
                objective='regression_l1',
                reg_alpha=0.8,
                reg_lambda=3.5,
                subsample=0.75,
                verbosity=-1,
                random_state=42
            ),
            
            'lgbm_2': LGBMRegressor(
                boosting_type='gbdt',
                colsample_bytree=0.85,
                learning_rate=0.05,
                max_depth=10,
                n_estimators=8000,
                num_leaves=40,
                objective='regression',
                reg_alpha=1.0,
                reg_lambda=4.0,
                subsample=0.8,
                verbosity=-1,
                random_state=123
            ),
            
            'xgb': XGBRegressor(
                learning_rate=0.05,
                max_depth=10,
                n_estimators=5000,
                subsample=0.8,
                colsample_bytree=0.9,
                reg_alpha=0.5,
                reg_lambda=2.0,
                objective='reg:absoluteerror',
                tree_method='hist',
                random_state=42
            ),
            
            'catboost': CatBoostRegressor(
                iterations=5000,
                learning_rate=0.05,
                depth=10,
                l2_leaf_reg=3.0,
                subsample=0.8,
                loss_function='MAE',
                verbose=False,
                random_state=42
            ),
            
            'extra_trees': ExtraTreesRegressor(
                n_estimators=500,
                max_depth=15,
                min_samples_split=10,
                min_samples_leaf=5,
                max_features='sqrt',
                n_jobs=-1,
                random_state=42
            )
        }
        
        return models
    
    def fit(self, X_train, y_train, verbose=True):
        self.models = self._create_models()
        for name, model in self.models.items():
            print(f"\n{'='*60}")
            print(f"Training {name.upper()}...")
            print(f"{'='*60}")
            
            try:
                model.fit(X_train, y_train)
                
                train_pred = model.predict(X_train)
                train_mape = mean_absolute_percentage_error(y_train, train_pred)
                train_mae = mean_absolute_error(y_train, train_pred)
                train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
                
                self.model_scores[name] = {
                    'train_mape': train_mape,
                    'train_mae': train_mae,
                    'train_rmse': train_rmse
                }
                
                if verbose:
                    print(f"\nTraining Metrics:")
                    print(f"  MAPE: {train_mape:.6f}")
                    print(f"  MAE:  {train_mae:.6f}")
                    print(f"  RMSE: {train_rmse:.6f}")
                
                self.weights[name] = 1.0 / (train_mape + 1e-8)
                
            except Exception as e:
                print(f"Error training {name}: {e}")
                self.weights[name] = 0.0
        
        total_weight = sum(self.weights.values())
        if total_weight > 0:
            self.weights = {k: v / total_weight for k, v in self.weights.items()}
        else:
            self.weights = {k: 1.0 / len(self.models) for k in self.models.keys()}
        
        for name, weight in sorted(self.weights.items(), key=lambda x: x[1], reverse=True):
            print(f"{name:15s}: {weight:.4f} (train_mape: {self.model_scores[name]['train_mape']:.6f})")
        
        ensemble_train_pred = self.predict(X_train)
        ensemble_train_mape = mean_absolute_percentage_error(y_train, ensemble_train_pred)
        ensemble_train_mae = mean_absolute_error(y_train, ensemble_train_pred)
        ensemble_train_rmse = np.sqrt(mean_squared_error(y_train, ensemble_train_pred))
        
        print(f"Training MAPE: {ensemble_train_mape:.6f}")
        print(f"Training MAE:  {ensemble_train_mae:.6f}")
        print(f"Training RMSE: {ensemble_train_rmse:.6f}")
        
        best_model = min(self.model_scores.items(), key=lambda x: x[1]['train_mape'])
        print(f"\nBest individual model: {best_model[0]} (MAPE: {best_model[1]['train_mape']:.6f})")
        print(f"Ensemble improvement: {((best_model[1]['train_mape'] - ensemble_train_mape) / best_model[1]['train_mape'] * 100):.2f}%")
        print("=" * 60)
        
        return self
    
    def predict(self, X):
        config = {
            "catboost": 0.4,
            "lgbm_1": 0.3,
            "lgbm_2": 0.3,
            "xgb": 0.5,
            "extra_trees": 0.4
        }
        
        predictions_dict = {}
        for name, model in self.models.items():
            predictions_dict[name] = model.predict(X)
        
        final = np.zeros(len(X))
        total_weight = 0
        
        for model_name, weight in config.items():
            if model_name in predictions_dict:
                final += np.array(predictions_dict[model_name]) * weight
                total_weight += weight
        
        if total_weight == 0:
            raise ValueError("No valid models available for prediction")
        
        final /= total_weight
        
        return final
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f"\n✓ Ensemble saved to {filepath}")
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y, dataset_name="Test"):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f"\n{dataset_name} Set Evaluation:")
        print(f"MAPE: {mape:.6f}")
        print(f"MAE:  {mae:.6f}")
        print(f"RMSE: {rmse:.6f}")
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}
    
    def get_feature_importance(self, feature_names, top_n=20):
        importance_dict = {}
        
        for name, model in self.models.items():
            if hasattr(model, 'feature_importances_'):
                importance = model.feature_importances_
                weight = self.weights[name]
                
                for i, feat in enumerate(feature_names):
                    if feat not in importance_dict:
                        importance_dict[feat] = 0
                    importance_dict[feat] += importance[i] * weight
        
        sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nTop {top_n} Most Important Features:")
        print("=" * 60)
        for i, (feat, imp) in enumerate(sorted_importance[:top_n], 1):
            print(f"{i:2d}. {feat:30s}: {imp:.6f}")
        
        return sorted_importance


ensemble = HybridEnsemble()
ensemble.fit(train_x, train_y, verbose=True)

#ensemble.evaluate(X, y, dataset_name="Full Training")
ensemble.get_feature_importance(X.columns, top_n=25)
ensemble.save('Hybrid_Ensemble_Model.pkl')

HYBRID_ENSEMBLE_MODEL = joblib.load('Hybrid_Ensemble_Model.pkl')


Training LGBM_1...

Training Metrics:
  MAPE: 75850088.169148
  MAE:  0.001836
  RMSE: 0.004690

Training LGBM_2...

Training Metrics:
  MAPE: 5593123443.534783
  MAE:  0.007615
  RMSE: 0.010798

Training XGB...

Training Metrics:
  MAPE: 101592.971984
  MAE:  0.000001
  RMSE: 0.000016

Training CATBOOST...


In [ ]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))